In [ ]:
!pip uninstall -y peft accelerate transformers


Found existing installation: accelerate 0.25.0
Uninstalling accelerate-0.25.0:
  Successfully uninstalled accelerate-0.25.0
Found existing installation: transformers 4.36.2
Uninstalling transformers-4.36.2:
  Successfully uninstalled transformers-4.36.2


In [ ]:
!pip install \
  torch \
  transformers==4.36.2 \
  accelerate==0.25.0 \
  datasets==2.16.1 \
  safetensors==0.4.2 \
  scikit-learn \
  sentencepiece


  Using cached transformers-4.36.2-py3-none-any.whl.metadata (126 kB)
  Using cached accelerate-0.25.0-py3-none-any.whl.metadata (18 kB)
Using cached transformers-4.36.2-py3-none-any.whl (8.2 MB)
Using cached accelerate-0.25.0-py3-none-any.whl (265 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.2 which is incompatible.


In [ ]:
import os
import torch
import numpy as np

from datasets import load_dataset
from transformers import (
    CLIPProcessor,
    CLIPModel,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

from torch import nn
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score
)


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

output_dir = "/content/drive/MyDrive/CLIP_MirageNews"
os.makedirs(output_dir, exist_ok=True)

print("Output dir:", output_dir)


In [ ]:
dataset = load_dataset("anson-huang/mirage-news")
dataset


In [ ]:
model_name = "openai/clip-vit-base-patch32"

processor = CLIPProcessor.from_pretrained(model_name)
clip_model = CLIPModel.from_pretrained(model_name)


In [ ]:
def preprocess(example):
    inputs = processor(
        text=example["text"],
        images=example["image"],
        truncation=True,
        padding="max_length",
        max_length=77,
        return_tensors="pt"
    )

    return {
        "input_ids": inputs["input_ids"][0],
        "attention_mask": inputs["attention_mask"][0],
        "pixel_values": inputs["pixel_values"][0],
        "labels": example["label"]
    }


In [ ]:
encoded_dataset = dataset.map(
    preprocess,
    remove_columns=dataset["train"].column_names
)


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
class CLIPForMFND(nn.Module):
    def __init__(self, clip_model, num_labels=2):
        super().__init__()
        self.clip = clip_model
        self.classifier = nn.Linear(
            clip_model.config.projection_dim * 2,
            num_labels
        )

    def forward(self, input_ids, attention_mask, pixel_values, labels=None):
        outputs = self.clip(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values
        )

        text_embeds = outputs.text_embeds
        image_embeds = outputs.image_embeds

        fused = torch.cat([text_embeds, image_embeds], dim=1)
        logits = self.classifier(fused)

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}


In [ ]:
model = CLIPForMFND(clip_model).to(device)


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(probs, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )

    acc = accuracy_score(labels, preds)

    auc = roc_auc_score(labels, probs[:, 1])

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc
    }


In [ ]:
training_args = TrainingArguments(
    output_dir=output_dir,

    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,

    evaluation_strategy="epoch",   # 👈 ĐÚNG
    save_strategy="epoch",
    save_total_limit=2,

    save_safetensors=False,

    load_best_model_at_end=True,
    metric_for_best_model="f1",

    fp16=torch.cuda.is_available(),
    report_to="none"
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [ ]:
!ls -lh /content/drive/MyDrive/CLIP_MirageNews/checkpoint-1250


total 1.7G
-rw------- 1 root root 1.2G Feb  7 15:53 optimizer.pt
-rw------- 1 root root 578M Feb  8 07:44 pytorch_model.bin
-rw------- 1 root root  15K Feb  7 15:53 rng_state.pth
-rw------- 1 root root 1.5K Feb  7 15:53 scheduler.pt
-rw------- 1 root root 1.5K Feb  7 15:53 trainer_state.json
-rw------- 1 root root 5.1K Feb  7 15:53 training_args.bin


In [ ]:
trainer.train()
# trainer.train(resume_from_checkpoint=True)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Auc
3,0.200600,0.216285,0.934400,0.980531,0.886400,0.931092,0.990854
4,0.136800,0.173133,0.959600,0.957769,0.961600,0.959681,0.994183
5,0.107100,0.164896,0.962800,0.969181,0.956000,0.962545,0.994791


TrainOutput(global_step=3125, training_loss=0.08770358093261718, metrics={'train_runtime': 4968.1468, 'train_samples_per_second': 10.064, 'train_steps_per_second': 0.629, 'total_flos': 0.0, 'train_loss': 0.08770358093261718, 'epoch': 5.0})

In [ ]:
metrics = trainer.evaluate()
print(metrics)
trainer.save_model(output_dir)
processor.save_pretrained(output_dir)

print("Model saved to:", output_dir)


{'eval_loss': 0.16489636898040771, 'eval_accuracy': 0.9628, 'eval_precision': 0.9691808596918086, 'eval_recall': 0.956, 'eval_f1': 0.9625453080950463, 'eval_auc': 0.9947907199999999, 'eval_runtime': 305.182, 'eval_samples_per_second': 8.192, 'eval_steps_per_second': 0.259, 'epoch': 5.0}
Model saved to: /content/drive/MyDrive/CLIP_MirageNews


In [ ]:
# ─── 2. Setup ─────────────────────────────────────────────────────────────────
import os, time, functools
import torch
import numpy as np
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from transformers import CLIPProcessor, CLIPModel
from datasets import load_dataset

from google.colab import drive
drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/CLIP_MirageNews"
device   = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ─── 3. Model class (giống hệt lúc train) ────────────────────────────────────
class CLIPForMFND(nn.Module):
    def __init__(self, clip_model, num_labels=2):
        super().__init__()
        self.clip       = clip_model
        self.classifier = nn.Linear(clip_model.config.projection_dim * 2, num_labels)

    def forward(self, input_ids=None, attention_mask=None,
                pixel_values=None, labels=None):
        outputs = self.clip(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values
        )
        fused  = torch.cat([outputs.text_embeds, outputs.image_embeds], dim=1)
        logits = self.classifier(fused)
        loss   = nn.CrossEntropyLoss()(logits, labels) if labels is not None else None
        return {"loss": loss, "logits": logits}

# ─── 4. Load model từ Drive ───────────────────────────────────────────────────
model_name = "openai/clip-vit-base-patch32"
processor  = CLIPProcessor.from_pretrained(SAVE_DIR)

torch.load = functools.partial(torch.load, weights_only=False)
clip_model = CLIPModel.from_pretrained(model_name)
model      = CLIPForMFND(clip_model).to(device)

weight_path = os.path.join(SAVE_DIR, "pytorch_model.bin")
state_dict  = torch.load(weight_path, map_location=device)
model.load_state_dict(state_dict)
model.eval()
print("✅ Model loaded from Drive")

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

# ─── 5. Preprocess + Evaluate helpers ────────────────────────────────────────
def make_test_dataset(split_name):
    ds = load_dataset("anson-huang/mirage-news", split=split_name)
    def preprocess(example):
        inputs = processor(
            text=example["text"], images=example["image"],
            truncation=True, padding="max_length",
            max_length=77, return_tensors="pt"
        )
        return {
            "input_ids":      inputs["input_ids"][0],
            "attention_mask": inputs["attention_mask"][0],
            "pixel_values":   inputs["pixel_values"][0],
            "labels":         example["label"]
        }
    encoded = ds.map(preprocess, remove_columns=ds.column_names, desc=split_name)
    return encoded

def collate_fn(batch):
    return {k: torch.tensor(np.array([b[k] for b in batch])) for k in batch[0]}

def evaluate_split(dataset):
    loader = DataLoader(dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            out = model(
                input_ids      = batch["input_ids"].to(device),
                attention_mask = batch["attention_mask"].to(device),
                pixel_values   = batch["pixel_values"].to(device)
            )
            all_logits.append(out["logits"].cpu())
            all_labels.append(batch["labels"])
    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()
    probs  = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds  = np.argmax(probs, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {
        "accuracy":  accuracy_score(labels, preds),
        "precision": p, "recall": r, "f1": f1,
        "auc":       roc_auc_score(labels, probs[:, 1])
    }

# ─── 6. Test Evaluation (5 splits) ───────────────────────────────────────────
test_splits = {
    "ID  (test1_nyt_mj)":    "test1_nyt_mj",
    "OOD (test2_bbc_dalle)": "test2_bbc_dalle",
    "OOD (test3_cnn_dalle)": "test3_cnn_dalle",
    "OOD (test4_bbc_sdxl)":  "test4_bbc_sdxl",
    "OOD (test5_cnn_sdxl)":  "test5_cnn_sdxl",
}

print("\n📊 Test Evaluation — CLIP ViT-B/32 FFT (MIRAGE-News):")
print(f"{'Split':<28} {'Acc':>7} {'Prec':>7} {'Rec':>7} {'F1':>7} {'AUC':>8}")
print("-" * 65)
for name, split in test_splits.items():
    td = make_test_dataset(split)
    m  = evaluate_split(td)
    print(f"{name:<28} {m['accuracy']*100:>6.2f}% {m['precision']*100:>6.2f}% "
          f"{m['recall']*100:>6.2f}% {m['f1']*100:>6.2f}% {m['auc']:>8.5f}")

# ─── 7. Latency + VRAM ───────────────────────────────────────────────────────
dummy_image = Image.new("RGB", (224, 224), (128, 128, 128))
dummy_text  = "This is a sample news headline for inference measurement."
inp = processor(text=dummy_text, images=dummy_image,
                truncation=True, padding="max_length",
                max_length=77, return_tensors="pt")
input_ids      = inp["input_ids"].to(device)
attention_mask = inp["attention_mask"].to(device)
pixel_values   = inp["pixel_values"].to(device)

if device == "cuda":
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

with torch.no_grad():
    for _ in range(10):
        model(input_ids, attention_mask, pixel_values)

latencies = []
with torch.no_grad():
    for _ in range(100):
        if device == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        model(input_ids, attention_mask, pixel_values)
        if device == "cuda": torch.cuda.synchronize()
        latencies.append((time.perf_counter() - t0) * 1000)

vram = torch.cuda.max_memory_allocated() / 1024**2 if device == "cuda" else 0
print(f"\n{'='*40}")
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}")
print(f"Latency median  : {np.median(latencies):.2f} ms")
print(f"VRAM (peak)     : {vram:.1f} MB")
print(f"{'='*40}")

Mounted at /content/drive
Using device: cpu


The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

OOD (test3_cnn_dalle)         54.00%  66.67%  16.00%  25.81%  0.84443


test4_bbc_sdxl:   0%|          | 0/500 [00:00<?, ? examples/s]

OOD (test4_bbc_sdxl)          75.00%  91.95%  54.80%  68.67%  0.91002


test5_cnn_sdxl:   0%|          | 0/500 [00:00<?, ? examples/s]

OOD (test5_cnn_sdxl)          75.80%  86.86%  60.80%  71.53%  0.92202

Total params    : 151,279,363
Trainable params: 151,279,363
Latency median  : 337.28 ms
VRAM (peak)     : 0.0 MB
